In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from pyproj import Transformer
import libpysal.weights as lps_weights
from esda.moran import Moran, Moran_BV
import warnings
warnings.filterwarnings("ignore")

print("Libraries loaded.")


## Step 1 — Load Data

`cleaned_bound_spaces.csv` is the spatially joined file: each recreational facility row has been tagged with the census tract it falls in (`name`) and that tract's diabetes percentage (`VALUE3`).

In [ ]:
df = pd.read_csv(
    "https://raw.githubusercontent.com/seven143143/GGR376-Final-Project/main/cleaned_bound_spaces.csv"
)
df = df.dropna(subset=["name", "VALUE3"]).copy()

print(f"Rows: {len(df):,}  |  Unique census tracts: {df['name'].nunique()}")
df.head()


## Step 2 — Aggregate to Census Tract Level

Each row is one facility. Group by CT to get:
- **`num_spaces`** — facility count per tract
- **`diabetes_pct`** — diabetes % (VALUE3) for that tract
- **`lat` / `lon`** — mean facility coordinates as centroid proxy

In [ ]:
tract_summary = (
    df.groupby("name")
    .agg(
        num_spaces   = ("OBJECTID",  "count"),
        diabetes_pct = ("VALUE3",    "first"),
        lat          = ("Latitude",  "mean"),
        lon          = ("Longitude", "mean"),
    )
    .reset_index()
    .dropna(subset=["lat", "lon"])
    .reset_index(drop=True)
)

print(f"Census tracts: {len(tract_summary)}")
tract_summary.describe()


## Step 3 — Pearson Correlation & Scatter Plot

In [ ]:
x = tract_summary["num_spaces"]
y = tract_summary["diabetes_pct"]

corr, p_corr = stats.pearsonr(x, y)
print(f"Pearson r = {corr:.4f}  |  p = {p_corr:.4f}")

m, b = np.polyfit(x, y, 1)
x_line = np.linspace(x.min(), x.max(), 200)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(x, y, color="#4C8BB5", alpha=0.45, edgecolors="white",
           linewidths=0.3, s=30, label="Census tracts")
ax.plot(x_line, m * x_line + b, color="#E05C3A", linewidth=2,
        label=f"Best-fit line  (slope = {m:.4f})")

textstr = f"Pearson r = {corr:.3f}\np = {p_corr:.4f}\nn = {len(tract_summary)} tracts"
props = dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.85, edgecolor="#ccc")
ax.text(0.97, 0.95, textstr, transform=ax.transAxes, fontsize=10,
        va="top", ha="right", bbox=props)

ax.set_xlabel("Number of recreational spaces in census tract", fontsize=12)
ax.set_ylabel("Diabetes percentage (%)", fontsize=12)
ax.set_title("Recreational Spaces vs. Diabetes Percentage\nby Census Tract",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


## Step 4 — Spatial Weights Matrix

Project to **UTM Zone 17N** (metres, appropriate for Ontario) then build a **KNN (k=8)** row-standardised weight matrix.

In [ ]:
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32617", always_xy=True)
easting, northing = transformer.transform(
    tract_summary["lon"].values,
    tract_summary["lat"].values
)
coords = np.column_stack([easting, northing])

k = 8
W = lps_weights.KNN.from_array(coords, k=k)
W.transform = "r"

print(f"Observations  : {W.n}")
print(f"Neighbours (k): {k}")
print(f"Non-zero links: {W.nonzero}")


## Step 5 — Standardise Variables

In [ ]:
X_raw = tract_summary["diabetes_pct"].values.astype(float)
Y_raw = tract_summary["num_spaces"].values.astype(float)

X_z = (X_raw - X_raw.mean()) / X_raw.std(ddof=1)
Y_z = (Y_raw - Y_raw.mean()) / Y_raw.std(ddof=1)

print(f"Diabetes (X)   — mean: {X_raw.mean():.3f}%,  std: {X_raw.std(ddof=1):.3f}%")
print(f"Rec Spaces (Y) — mean: {Y_raw.mean():.1f},    std: {Y_raw.std(ddof=1):.1f}")


## Step 6 — Univariate Moran's I

Is each variable spatially clustered on its own?
- **I > 0** → similar values cluster together
- **I ≈ 0** → spatially random
- **I < 0** → dissimilar values cluster (checkerboard)

In [ ]:
np.random.seed(42)
PERMS = 999

mi_x = Moran(X_z, W, permutations=PERMS)
mi_y = Moran(Y_z, W, permutations=PERMS)

print("=" * 60)
print("UNIVARIATE MORAN'S I")
print("=" * 60)
print(f"{'Variable':<28} {'I':>7}  {'E[I]':>7}  {'p-value':>8}  {'Sig?':>5}")
print("-" * 60)
print(f"{'Diabetes %':<28} {mi_x.I:>7.4f}  {mi_x.EI:>7.4f}  {mi_x.p_sim:>8.4f}  {'Yes' if mi_x.p_sim < 0.05 else 'No':>5}")
print(f"{'Recreational Spaces':<28} {mi_y.I:>7.4f}  {mi_y.EI:>7.4f}  {mi_y.p_sim:>8.4f}  {'Yes' if mi_y.p_sim < 0.05 else 'No':>5}")
print(f"\n(999 permutations, alpha = 0.05)")


## Step 7 — Bivariate Moran's I

Measures the spatial association between **diabetes at location *i*** and the **spatial lag of recreational spaces at neighbouring locations *j***.

$$I_{BV} = \frac{\sum_i z_{x,i} \sum_j w_{ij} z_{y,j}}{n}$$

- **I > 0** → high-diabetes tracts are surrounded by tracts with *more* rec spaces
- **I < 0** → high-diabetes tracts are surrounded by tracts with *fewer* rec spaces

In [ ]:
np.random.seed(42)
bv = Moran_BV(X_z, Y_z, W, permutations=PERMS)

print("=" * 60)
print("BIVARIATE MORAN'S I  (X = Diabetes, Y = Rec Spaces)")
print("=" * 60)
print(f"  Moran's I  = {bv.I:.4f}")
print(f"  z-score    = {bv.z_sim:.4f}")
print(f"  p-value    = {bv.p_sim:.4f}  ({PERMS} permutations)")
print(f"  Significant: {'Yes' if bv.p_sim < 0.05 else 'No'}  (alpha = 0.05)")

if bv.p_sim < 0.05:
    if bv.I < 0:
        print()
        print("  Interpretation: Significant NEGATIVE spatial autocorrelation.")
        print("  Census tracts with HIGH diabetes tend to be surrounded by tracts")
        print("  with FEWER recreational spaces — consistent with spatial inequity")
        print("  where underserved (high-diabetes) areas have lower nearby access.")
    else:
        print()
        print("  Interpretation: Significant POSITIVE spatial autocorrelation.")
        print("  Census tracts with HIGH diabetes tend to cluster near tracts")
        print("  with MORE recreational spaces.")
else:
    print()
    print("  No statistically significant spatial co-clustering detected.")


## Step 8 — Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(
    f"Bivariate Moran's I — Diabetes (X) vs Recreational Spaces (Y)\n"
    f"I = {bv.I:.4f}  |  z = {bv.z_sim:.4f}  |  p = {bv.p_sim:.4f}",
    fontsize=13, fontweight="bold", y=1.02
)

# ── Plot 1: Bivariate Moran scatter ──────────────────────────────────────────
ax = axes[0]
Wy = W.sparse.dot(Y_z)

quad_colors = []
for xi, wyi in zip(X_z, Wy):
    if   xi >= 0 and wyi >= 0: quad_colors.append("#D73027")
    elif xi <  0 and wyi <  0: quad_colors.append("#4575B4")
    elif xi >= 0 and wyi <  0: quad_colors.append("#FC8D59")
    else:                       quad_colors.append("#91BFDB")

ax.scatter(X_z, Wy, c=quad_colors, alpha=0.45, edgecolors="white",
           linewidths=0.2, s=15)
xl = np.linspace(X_z.min(), X_z.max(), 200)
slope, intercept = np.polyfit(X_z, Wy, 1)
ax.plot(xl, slope * xl + intercept, "k--", lw=1.5, label=f"slope = {slope:.3f}")
ax.axhline(0, color="grey", lw=0.7)
ax.axvline(0, color="grey", lw=0.7)
patches = [
    mpatches.Patch(color="#D73027", label="HH"),
    mpatches.Patch(color="#4575B4", label="LL"),
    mpatches.Patch(color="#FC8D59", label="HL"),
    mpatches.Patch(color="#91BFDB", label="LH"),
]
ax.legend(handles=patches, fontsize=8, loc="upper right", title="Quadrant")
ax.set_xlabel("Diabetes % (z-score)", fontsize=10)
ax.set_ylabel("Spatial Lag — Rec Spaces (z-score)", fontsize=10)
ax.set_title(f"Bivariate Moran Scatter\nI = {bv.I:.4f}", fontsize=10)

# ── Plot 2: Permutation distribution ─────────────────────────────────────────
ax = axes[1]
ax.hist(bv.sim, bins=50, color="#A8C6E0", edgecolor="white", density=True)
ax.axvline(bv.I, color="#D73027", lw=2.5, label=f"Observed I = {bv.I:.4f}")
ax.axvline(np.mean(bv.sim), color="#555", lw=1.5, ls="--",
           label=f"Mean sim = {np.mean(bv.sim):.4f}")
ax.set_xlabel("Simulated Moran's I", fontsize=10)
ax.set_ylabel("Density", fontsize=10)
ax.set_title("Permutation Reference Distribution\n(999 permutations)", fontsize=10)
ax.legend(fontsize=9)

# ── Plot 3: Univariate Moran scatter (diabetes) ───────────────────────────────
ax = axes[2]
Wx = W.sparse.dot(X_z)
ax.scatter(X_z, Wx, color="#5B8DB8", alpha=0.4, edgecolors="white",
           linewidths=0.2, s=15)
m2, b2 = np.polyfit(X_z, Wx, 1)
ax.plot(xl, m2 * xl + b2, color="#E05C3A", lw=1.8,
        label=f"slope = {m2:.3f}  (I = {mi_x.I:.3f})")
ax.axhline(0, color="grey", lw=0.7)
ax.axvline(0, color="grey", lw=0.7)
ax.set_xlabel("Diabetes % (z-score)", fontsize=10)
ax.set_ylabel("Spatial Lag — Diabetes (z-score)", fontsize=10)
ax.set_title(f"Univariate Moran Scatter (Diabetes)\nI = {mi_x.I:.4f},  p = {mi_x.p_sim:.4f}",
             fontsize=10)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()


## Summary

In [ ]:
summary = pd.DataFrame({
    "Test": [
        "Pearson r (aspatial)",
        "Univariate Moran's I — Diabetes",
        "Univariate Moran's I — Rec Spaces",
        "Bivariate Moran's I (X=Diabetes, Y=Rec Spaces)",
    ],
    "Statistic": [
        f"r = {corr:.4f}", f"I = {mi_x.I:.4f}",
        f"I = {mi_y.I:.4f}", f"I = {bv.I:.4f}"
    ],
    "p-value": [
        f"{p_corr:.4f}", f"{mi_x.p_sim:.4f}",
        f"{mi_y.p_sim:.4f}", f"{bv.p_sim:.4f}"
    ],
    "Sig. (a=0.05)": [
        "Yes" if p_corr    < 0.05 else "No",
        "Yes" if mi_x.p_sim < 0.05 else "No",
        "Yes" if mi_y.p_sim < 0.05 else "No",
        "Yes" if bv.p_sim   < 0.05 else "No",
    ]
})
summary
